## Post-processing COMSOL solution

In [5]:
import time
from pathlib import Path
from typing import Any
import dataclasses
import shutil
import numpy as np
from scipy.spatial import KDTree
import pyvista as pv

#%%
# Set file locations
case_name = "case2"
mesh_name = "coil_box_named.msh"

mesh_path = "../../meshes/"
output_path = f"../../output/{case_name}/{case_name}_comsol"
output_path_gauss = f"../../output/{case_name}/gauss/{case_name}_comsol"
mesh_file_path = mesh_path + mesh_name

In [6]:
sim_data = pv.read(output_path + "_export" + ".vtu")

# Extract data for the cells corresponding to the coil.
# sim_data = sim_data.extract_values(values=1, scalars="Material_settings", preference="point")

In [7]:
# elec_pot_unordered = sim_data["Electric_potential"]
mag_vec_unordered_X = sim_data["Magnetic_vector_potential,_x-component"][:, np.newaxis]
mag_vec_unordered_Y = sim_data["Magnetic_vector_potential,_y-component"][:, np.newaxis]
mag_vec_unordered_Z = sim_data["Magnetic_vector_potential,_z-component"][:, np.newaxis]

mag_flux_unordered_X = sim_data["Magnetic_flux_density,_x-component"][:, np.newaxis]
mag_flux_unordered_Y = sim_data["Magnetic_flux_density,_y-component"][:, np.newaxis]
mag_flux_unordered_Z = sim_data["Magnetic_flux_density,_z-component"][:, np.newaxis]

# sol1_unordered = elec_pot_unordered
sol2_unordered = np.concatenate((mag_vec_unordered_X, mag_vec_unordered_Y, mag_vec_unordered_Z), axis=1)
sol3_unordered = np.concatenate((mag_flux_unordered_X, mag_flux_unordered_Y, mag_flux_unordered_Z), axis=1)
# print(sol1_unordered.shape)
print(sol2_unordered.shape)
print(sol3_unordered.shape)
# print(attribute.shape)

# points_unordered = sim_data.points
# res = pv.read(mesh_file_path)
# points = res.points
# tree2 = KDTree(points_unordered)

# sol1 = np.zeros(len(points))
# sol2 = np.zeros((len(points), 3))
# sol3 = np.zeros((len(points), 3))

# for i in range(len(points)):
#     _, index = tree2.query(points[i, :], distance_upper_bound=1e-9)
#     if index == tree2.n:
#         continue
#     else:
#         # sol1[i] = sol1_unordered[index]
#         sol2[i, :] = sol2_unordered[index]
#         sol3[i, :] = sol3_unordered[index]

# res["electric_potential"] = sol1
# res["magnetic_vector_potential"] = sol2
# res["magnetic_flux_density"] = sol3

sim_data["magnetic_vector_potential"] = sol2_unordered
sim_data["magnetic_flux_density"] = sol3_unordered
# sim_data.rename_array('Electric_potential', 'electric_potential')


sim_data.point_data.remove("Magnetic_vector_potential,_x-component")
sim_data.point_data.remove("Magnetic_vector_potential,_y-component")
sim_data.point_data.remove("Magnetic_vector_potential,_z-component")

sim_data.point_data.remove("Magnetic_flux_density,_x-component")
sim_data.point_data.remove("Magnetic_flux_density,_y-component")
sim_data.point_data.remove("Magnetic_flux_density,_z-component")

sim_data.save(output_path + ".vtu")

(278516, 3)
(278516, 3)


In [8]:
results_gauss = np.genfromtxt(output_path_gauss + '.txt', delimiter=' ')

print(results_gauss.shape)

(69629, 11)
